In [7]:
!python generate_data.py

All datasets generated successfully!
Duka sales records: 1000
Agent records: 20
Agent transaction records: 612


In [8]:
import os

os.listdir('/content')

['.config',
 'agents.csv',
 'generate_data.py',
 'duka_sales.csv',
 'agent_transactions.csv',
 'sample_data']

In [9]:
!python3 /content/generate_data.py

All datasets generated successfully!
Duka sales records: 1000
Agent records: 20
Agent transaction records: 612


In [10]:
import pandas as pd
import numpy as np

# Load Duka sales data
sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

# Display first five records
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



# Executive Summary

- Nairobi branch generated KES 26,106,100 in revenue, ranking first among the 5 branches.
- Laptops generated the highest product revenue of KES 41,535,000, while tablets led unit sales with 691 units.
- Weekday orders totalled 723, generating KES 51,856,200 compared with 277 weekend orders generating KES 19,936,300.
- April was the best sales month at KES 13,518,300, while June was the lowest at KES 10,903,400, a 24.03% difference relative to June.
- There were 83 Premium online orders from Nairobi and Mombasa, generating KES 10,218,000 in revenue.



## 1. First-Five-Minutes Data Inspection

Before analysing the sales data, I inspected its structure, quality, date coverage and key categories.

The inspection covers:
- Dataset shape and number of records.
- Column names and data types.
- Missing values.
- Descriptive statistics.
- Unique values in each column.
- Branch and sales channel distribution.
- Earliest and latest order dates.


In [11]:

# First five records
display(sales.head())

# Last five records
display(sales.tail())

# Dataset shape
print("Dataset shape:", sales.shape)

# Column information and data types
sales.info()

# Descriptive statistics
display(sales.describe())

# Missing values
print("\nMissing values:")
display(sales.isnull().sum())

# Unique values per column
print("\nUnique values:")
display(sales.nunique())

# Branch distribution
print("\nBranch distribution:")
display(sales["branch"].value_counts(normalize=True) * 100)

# Channel distribution
print("\nSales channel distribution:")
display(sales["channel"].value_counts(normalize=True) * 100)

# Date range
print("\nEarliest order:", sales["order_date"].min())
print("Latest order:", sales["order_date"].max())


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


Dataset shape: (1000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546



Missing values:


,0
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0


... (3 outputs left out)

,proportion
branch,
Nairobi,37.6
Mombasa,18.6
Nakuru,16.2
Kisumu,15.9
Eldoret,11.7



Sales channel distribution:


,proportion
channel,
In-store,57.9
Online,32.0
Phone order,10.1



Earliest order: 2026-01-01 00:00:00
Latest order: 2026-06-29 00:00:00



### Inspection Observations

1. The dataset contains 1,000 rows and 7 columns.
2. The order_date column is stored as a datetime type, while the categorical columns contain text.
3. The dataset contains no missing values.
4. The order dates range from 1 January 2026 to 29 June 2026.
5. The dataset contains five branches, five products and three sales channels.
6. Nairobi has the largest share of orders, while Eldoret has the smallest share.
7. In-store is the most frequently used sales channel.



## 2. Feature Engineering

New columns are created to make sales analysis easier.

The derived columns are:
- revenue: quantity multiplied by unit price.
- month: month in which the order was placed.
- weekday: day of the week.
- is_weekend: identifies Saturday and Sunday orders.
- price_band: Premium for unit prices of KES 28,000 or above, and Value otherwise.


In [12]:

# Calculate order revenue
sales["revenue"] = sales["quantity"] * sales["unit_price"]

# Extract month name
sales["month"] = sales["order_date"].dt.month_name()

# Extract weekday name
sales["weekday"] = sales["order_date"].dt.day_name()

# Identify weekend orders
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

# Categorise products by price
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

# Display updated dataset
display(sales.head())

# Check derived columns
display(sales[[
    "revenue",
    "month",
    "weekday",
    "is_weekend",
    "price_band"
]].describe(include="all"))


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


,revenue,month,weekday,is_weekend,price_band
count,1000.00000,1000,1000,1000,1000
unique,NaN,6,7,2,2
top,NaN,March,Friday,False,Value
freq,NaN,186,166,723,565
mean,71792.50000,NaN,NaN,NaN,NaN
std,84180.91827,NaN,NaN,NaN,NaN
min,1200.00000,NaN,NaN,NaN,NaN
25%,6000.00000,NaN,NaN,NaN,NaN
50%,45000.00000,NaN,NaN,NaN,NaN
75%,112000.00000,NaN,NaN,NaN,NaN



## Task 3: Branch Performance Ranking

The objective is to compare branch performance using total revenue and number of orders.


In [13]:

# Rank branches by total revenue
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Rank branches by number of orders
branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("Branch ranking by revenue:")
display(branch_revenue)

print("Branch ranking by number of orders:")
display(branch_orders)

# Compare rankings
comparison = pd.DataFrame({
    "Total Revenue": branch_revenue,
    "Number of Orders": branch_orders
})

comparison["Revenue Rank"] = comparison["Total Revenue"].rank(
    ascending=False
)
comparison["Order Rank"] = comparison["Number of Orders"].rank(
    ascending=False
)

display(comparison)


Branch ranking by revenue:


,revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


Branch ranking by number of orders:


,order_id
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


,Total Revenue,Number of Orders,Revenue Rank,Order Rank
branch,,,,
Nairobi,26106100,376,1.0,1.0
Mombasa,14003700,186,2.0,2.0
Nakuru,11970400,162,3.0,3.0
Kisumu,10868100,159,4.0,4.0
Eldoret,8844200,117,5.0,5.0



### Answer and Interpretation

The revenue ranking and order-volume ranking are the same.

Nairobi ranks first in both total revenue and number of orders, while Eldoret ranks last. This indicates that branches with more orders also generated more total revenue in this dataset.

However, revenue is also influenced by the products sold and their prices, not just the number of orders.



## Task 4: Top Three Products by Revenue and Units Sold

This analysis identifies the three products generating the highest revenue and the three products with the highest total units sold.


In [14]:

# Top three products by revenue
top_revenue = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

# Top three products by units sold
top_units = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("Top three products by revenue:")
display(top_revenue)

print("Top three products by units sold:")
display(top_units)


Top three products by revenue:


,revenue
product,
Laptop,41535000
Tablet,19348000
Phone,8880000


Top three products by units sold:


,quantity
product,
Tablet,691
Laptop,639
Phone,592



### Answer and Interpretation

Laptops lead in revenue, generating KES 41,535,000, followed by tablets at KES 19,348,000 and phones at KES 8,880,000.

Tablets lead in units sold with 691 units, followed by laptops with 639 units and phones with 592 units.

The rankings differ because laptops have a higher unit price than tablets. Therefore, laptops generate more revenue even though fewer units were sold.



## Task 5: Weekend and Weekday Sales Comparison

This section compares the number of orders, total revenue and average order revenue for weekdays and weekends.


In [15]:

weekend_comparison = sales.groupby("is_weekend").agg(
    number_of_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

weekend_comparison.index = [
    "Weekday" if not value else "Weekend"
    for value in weekend_comparison.index
]

display(weekend_comparison.round(2))


,number_of_orders,total_revenue,average_order_revenue
Weekday,723,51856200,71723.65
Weekend,277,19936300,71972.20



### Answer and Interpretation

Weekdays recorded 723 orders and generated KES 51,856,200 in revenue. Weekends recorded 277 orders and generated KES 19,936,300.

The average order revenue was approximately KES 71,724 on weekdays and KES 71,972 on weekends.

Weekdays generated more total revenue because they had considerably more orders. However, the average order revenue was slightly higher on weekends.



## Task 6: Monthly Revenue Performance

This section identifies the month with the highest and lowest revenue and calculates the percentage difference between them.


In [16]:

# Calculate monthly revenue in calendar order
monthly_revenue = (
    sales.groupby(sales["order_date"].dt.month)["revenue"]
    .sum()
)

monthly_revenue.index = pd.to_datetime(
    monthly_revenue.index,
    format="%m"
).month_name()

display(monthly_revenue)

# Identify best and worst months
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

# Percentage difference relative to the worst month
percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("Best month:", best_month)
print("Best revenue: KES", best_revenue)

print("Worst month:", worst_month)
print("Worst revenue: KES", worst_revenue)

print("Percentage difference:", round(percentage_difference, 2), "%")


,revenue
order_date,
January,11786600
February,11772000
March,12396800
April,13518300
May,11415400
June,10903400


Best month: April
Best revenue: KES 13518300
Worst month: June
Worst revenue: KES 10903400
Percentage difference: 23.98 %



### Answer and Interpretation

April was the best-performing month, generating KES 13,518,300 in revenue.

June was the worst-performing month, generating KES 10,903,400.

April's revenue was approximately 24.03% higher than June's revenue, using June as the comparison base.

The business should investigate the factors behind the decline in June revenue and identify opportunities to improve sales performance.



## Task 7: Premium Online Orders from Nairobi and Mombasa

The objective is to identify Premium orders placed through the Online channel in Nairobi and Mombasa, calculate their number and total revenue, and export the filtered data.


In [17]:

# Filter Premium online orders from Nairobi and Mombasa
premium_online_coast_capital = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Calculate number of orders
number_of_orders = len(premium_online_coast_capital)

# Calculate total revenue
total_revenue = premium_online_coast_capital["revenue"].sum()

print("Number of Premium online orders:", number_of_orders)
print("Total revenue: KES", total_revenue)

# Display filtered data
display(premium_online_coast_capital.head())

# Export without index
from pathlib import Path

# Create outputs folder if it does not exist
Path("outputs").mkdir(exist_ok=True)

# Save filtered data in the correct location
premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("Filtered dataset exported successfully.")


Number of Premium online orders: 83
Total revenue: KES 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


Filtered dataset exported successfully.



### Answer and Interpretation

The analysis identified 83 Premium online orders from Nairobi and Mombasa.

These orders generated total revenue of KES 10,218,000.

The filtered dataset was exported as premium_online_coast_capital.csv without the DataFrame index.


In [18]:
from google.colab import files

files.download(
    "outputs/premium_online_coast_capital.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>